In [2]:
import pandas as pd

In [3]:
# Loading the dataset using pandas
df=pd.read_csv(r"C:\Users\Dell\Downloads\customer_shopping_behavior.csv")

In [4]:
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [5]:
# Checking the stats of the dataset
df.describe()

,Customer ID,Age,Purchase Amount (USD),Review Rating,Previous Purchases
count,3900.000000,3900.000000,3900.000000,3863.000000,3900.000000
mean,1950.500000,44.068462,59.764359,3.750065,25.351538
std,1125.977353,15.207589,23.685392,0.716983,14.447125
min,1.000000,18.000000,20.000000,2.500000,1.000000
25%,975.750000,31.000000,39.000000,3.100000,13.000000
50%,1950.500000,44.000000,60.000000,3.800000,25.000000
75%,2925.250000,57.000000,81.000000,4.400000,38.000000
max,3900.000000,70.000000,100.000000,5.000000,50.000000


In [6]:
# Checking for the null values
df.isnull().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [7]:
# Median is a better choice for handling missing values
# Since it handles the outliers 
# Here category wise median will be a better choice

In [8]:
# Imputing missing values
df["Review Rating"]=df.groupby("Item Purchased")["Review Rating"].transform(lambda x:x.fillna(x.median()))

In [9]:
df.isnull().sum()

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

In [10]:
# Creating a new column Age Group
groups = ['Young Adult', 'Adult', 'Middle-aged','Senior']
df['Age Group'] = pd.qcut(df['Age'], q=4, labels= groups)

In [11]:
df[["Age","Age Group"]].head(20)

,Age,Age Group
0,55,Middle-aged
1,19,Young Adult
2,50,Middle-aged
3,21,Young Adult
4,45,Middle-aged
5,46,Middle-aged
6,63,Senior
7,27,Young Adult
8,26,Young Adult
9,57,Middle-aged


In [12]:
# Mapping the Frequency of Purchases with number of days for better analysis
mapping = {
    "Fortnightly" : 14,
    "Weekly" : 7,
    "Monthly" : 30,
    "Quarterly" : 90,
    "Bi-Weekly" : 14,
    "Annually" : 365,
    "Every 3 Months" : 90
}

df["purchase_frequency_days"]= df["Frequency of Purchases"].map(mapping)

In [13]:
df[["purchase_frequency_days", "Frequency of Purchases"]].head(10)

,purchase_frequency_days,Frequency of Purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually
5,7,Weekly
6,90,Quarterly
7,7,Weekly
8,365,Annually
9,90,Quarterly


In [14]:
# Checking whether the Discount Applied and Promo Code Used can be combined in case they have same values throughout
(df["Discount Applied"] == df["Promo Code Used"]).all()

np.True_

In [15]:
# Dropping the promo code used since both cols have same values
df= df.drop("Promo Code Used", axis=1)

In [16]:
# Checking the execution of drop command
df.columns

Index(['Customer ID', 'Age', 'Gender', 'Item Purchased', 'Category',
       'Purchase Amount (USD)', 'Location', 'Size', 'Color', 'Season',
       'Review Rating', 'Subscription Status', 'Shipping Type',
       'Discount Applied', 'Previous Purchases', 'Payment Method',
       'Frequency of Purchases', 'Age Group', 'purchase_frequency_days'],
      dtype='object')

In [17]:
df.columns = df.columns.str.lower().str.replace(" ","_")
df.rename(columns={"purchase_amount_(usd)":"purchase_amount"}, inplace=True)
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='object')

In [18]:
import sqlite3

conn = sqlite3.connect("Customers.db")
df.to_sql("customer", conn, if_exists="replace", index=False)
df.to_csv("customer.csv", index=False)

In [18]:
# Creating a function for execution of sql queries
def run_query(query):
    return pd.read_sql_query(query, conn)

In [19]:
run_query("Select * from customer;")

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age_group,purchase_frequency_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly,Middle-aged,14
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly,Young Adult,14
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly,Middle-aged,7
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,49,PayPal,Weekly,Young Adult,7
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,31,PayPal,Annually,Middle-aged,365
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3895,3896,40,Female,Hoodie,Clothing,28,Virginia,L,Turquoise,Summer,4.2,No,2-Day Shipping,No,32,Venmo,Weekly,Adult,7
3896,3897,52,Female,Backpack,Accessories,49,Iowa,L,White,Spring,4.5,No,Store Pickup,No,41,Bank Transfer,Bi-Weekly,Middle-aged,14
3897,3898,46,Female,Belt,Accessories,33,New Jersey,L,Green,Spring,2.9,No,Standard,No,24,Venmo,Quarterly,Middle-aged,90
3898,3899,44,Female,Shoes,Footwear,77,Minnesota,S,Brown,Summer,3.8,No,Express,No,24,Venmo,Weekly,Adult,7


In [20]:
df.columns.tolist()

['customer_id',
 'age',
 'gender',
 'item_purchased',
 'category',
 'purchase_amount',
 'location',
 'size',
 'color',
 'season',
 'review_rating',
 'subscription_status',
 'shipping_type',
 'discount_applied',
 'previous_purchases',
 'payment_method',
 'frequency_of_purchases',
 'age_group',
 'purchase_frequency_days']

### 1. What is the total revenue generated by male vs. female customers?

In [21]:
q1 = run_query(
    """Select gender, SUM(purchase_amount) as total_revenue
    from customer group by gender;""")
q1

,gender,total_revenue
0,Female,75191
1,Male,157890


### 2. Which customers used a discount but still spent more than the average purchase amount?

In [22]:
q2 = run_query("""
    Select customer_id, purchase_amount
    from customer
    where discount_applied="Yes" and purchase_amount >(Select AVG(purchase_amount) from customer);""")
q2

,customer_id,purchase_amount
0,2,64
1,3,73
2,4,90
3,7,85
4,9,97
...,...,...
834,1667,64
835,1671,73
836,1673,73
837,1674,62


### 3. Which are the top 5 products with the highest average review rating?

In [23]:
q3 = run_query("""
    Select item_purchased, round(Avg(review_rating),2) avg_review_rating
    from customer
    group by item_purchased
    order by avg_review_rating desc
    limit 5;""")
q3

,item_purchased,avg_review_rating
0,Gloves,3.86
1,Sandals,3.85
2,Boots,3.82
3,Hat,3.80
4,Skirt,3.79


### 4. Compare the average Purchase Amounts between Standard and Express Shipping.

In [24]:
q4 = run_query("""
    Select shipping_type, round(avg(purchase_amount),2) as avg_purchase_amount
    from customer
    where shipping_type in ("Standard", "Express")
    group by shipping_type;""")
q4

,shipping_type,avg_purchase_amount
0,Express,60.48
1,Standard,58.46


### 5. Do subscribed customers spend more? Compare average spend and total revenue between subscribers and non-subscribers.

In [25]:
q5 = run_query("""
    Select subscription_status, 
    count(customer_id) as total_customers,
    round(avg(purchase_amount),2) as avg_spend,
    round(sum(purchase_amount),2) as total_revenue
    from customer
    group by subscription_status;""")
q5

,subscription_status,total_customers,avg_spend,total_revenue
0,No,2847,59.87,170436.0
1,Yes,1053,59.49,62645.0


### 6. Which 5 products have the highest percentage of purchases with discounts applied?

In [26]:
q6 = run_query("""
    Select item_purchased, 
    round(100 * sum(case when discount_applied="Yes" then 1 else 0 end)/count(*)) as discount_rate
    from customer
    group by item_purchased
    order by discount_rate DESC
    limit 5;""")
q6

,item_purchased,discount_rate
0,Hat,50.0
1,Sneakers,49.0
2,Coat,49.0
3,Sweater,48.0
4,Pants,47.0


### 7. Segment customers into new, returning, and loyal based on their total number of previous purchases and show the count of each segment.

In [27]:
q7 = run_query("""
    with type as (Select customer_id, previous_purchases,
    case when previous_purchases = 1 then "New"
         when previous_purchases BETWEEN 2 and 10 then "Returning"
         else "Loyal" end as customer_type
    from customer)
    
    Select customer_type, count(*) as customer_count
    from type
    group by customer_type;""")
q7

,customer_type,customer_count
0,Loyal,3116
1,New,83
2,Returning,701


### 8. What are the top 3 most purchased products within each category?

In [33]:
q8 = run_query("""
    With item_counts as(
        Select category, item_purchased,
        count(customer_id) as total_orders,
        ROW_NUMBER() over (partition by category order by count(customer_id) DESC)
        as item_rank
        from customer
        group by category, item_purchased
    )
    Select item_rank, category, item_purchased, total_orders
    from item_counts
    where item_rank <=3;""")
q8

,item_rank,category,item_purchased,total_orders
0,1,Accessories,Jewelry,171
1,2,Accessories,Sunglasses,161
2,3,Accessories,Belt,161
3,1,Clothing,Pants,171
4,2,Clothing,Blouse,171
5,3,Clothing,Shirt,169
6,1,Footwear,Sandals,160
7,2,Footwear,Shoes,150
8,3,Footwear,Sneakers,145
9,1,Outerwear,Jacket,163


### 9. Are customers who are repeat buyers (more than 5 previous purchases) also likely to subscribe?

In [35]:
q9 = run_query("""
    Select subscription_status, count(customer_id) as repeat_buyers
    from customer 
    where previous_purchases >5
    group by subscription_status;""")
q9

,subscription_status,repeat_buyers
0,No,2518
1,Yes,958


### 10. What is the revenue contribution of each age group?

In [36]:
q10 = run_query("""
    Select age_group,
    sum(purchase_amount) as total_revenue
    from customer
    group by age_group
    order by total_revenue desc;""")
q10

,age_group,total_revenue
0,Young Adult,62143
1,Middle-aged,59197
2,Adult,55978
3,Senior,55763
